In [1]:
import json
import pandas as pd
from collections import Counter
with open ("raw_studies.json") as f:
    all_studies = json.load(f)
df_int = pd.read_csv("../data/clean/studies.csv")
int_ids = set(df_int["nctId"])

print(len(all_studies), len(df_int), len(int_ids))

3721 1717 1717


In [2]:
# checking if browseBranches module exists - it does not
s = all_studies[0]

print(s["derivedSection"].keys())
print(s["derivedSection"]["conditionBrowseModule"].keys())
#print(s["derivedSection"]["conditionBrowseModule"]["browseBranches"])
print(s["protocolSection"]["conditionsModule"]["conditions"])

dict_keys(['miscInfoModule', 'conditionBrowseModule', 'interventionBrowseModule'])
dict_keys(['meshes', 'ancestors'])
['Dermatitis', 'Dermatitis, Atopic', 'Eczema', 'Skin Diseases', 'Skin Diseases, Genetic', 'Genetic Diseases, Inborn', 'Skin Diseases, Eczematous', 'Hypersensitivity', 'Hypersensitivity, Immediate', 'Immune System Diseases']


In [3]:
#how data looks like on a study. MeSH terms of diseases & broader categ
s = all_studies[0]
cbm = s["derivedSection"]["conditionBrowseModule"]

print(cbm["meshes"])
print(cbm["ancestors"])
print(s["protocolSection"]["conditionsModule"]["conditions"])

[{'id': 'D003872', 'term': 'Dermatitis'}, {'id': 'D003876', 'term': 'Dermatitis, Atopic'}, {'id': 'D004485', 'term': 'Eczema'}, {'id': 'D012871', 'term': 'Skin Diseases'}, {'id': 'D012873', 'term': 'Skin Diseases, Genetic'}, {'id': 'D030342', 'term': 'Genetic Diseases, Inborn'}, {'id': 'D017443', 'term': 'Skin Diseases, Eczematous'}, {'id': 'D006967', 'term': 'Hypersensitivity'}, {'id': 'D006969', 'term': 'Hypersensitivity, Immediate'}, {'id': 'D007154', 'term': 'Immune System Diseases'}]
[{'id': 'D017437', 'term': 'Skin and Connective Tissue Diseases'}, {'id': 'D009358', 'term': 'Congenital, Hereditary, and Neonatal Diseases and Abnormalities'}]
['Dermatitis', 'Dermatitis, Atopic', 'Eczema', 'Skin Diseases', 'Skin Diseases, Genetic', 'Genetic Diseases, Inborn', 'Skin Diseases, Eczematous', 'Hypersensitivity', 'Hypersensitivity, Immediate', 'Immune System Diseases']


In [4]:
#types of dict keys for all those 1717 studies
key_counts = Counter()
n_total = 0
for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    n_total+=1
    cbm = s.get("derivedSection", {}).get("conditionBrowseModule", {})
    key_counts.update(cbm.keys())

print(n_total)
print(key_counts)

1717
Counter({'meshes': 1523, 'ancestors': 1501})


In [5]:
#most common diseases & categories. 194 studies have no terms at all (1717 studies - 1523 studies with terms)

mesh_counts = Counter() # specific dieeases
anc_counts = Counter() # broader categories
n_with_mesh = 0

for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    cbm = s.get("derivedSection", {}).get("conditionBrowseModule", {})
    meshes = [m["term"] for m in cbm.get("meshes", [])]
    ancestors = [a["term"] for a in cbm.get("ancestors", [])]
    if meshes:
        n_with_mesh += 1
    mesh_counts.update(meshes)
    anc_counts.update(ancestors)

print(n_with_mesh)
print(anc_counts.most_common(30))
print(mesh_counts.most_common(30))

1523
[('Neoplasms', 386), ('Pathological Conditions, Signs and Symptoms', 364), ('Neoplasms by Site', 323), ('Respiratory Tract Diseases', 291), ('Immune System Diseases', 277), ('Lung Diseases', 269), ('Pathologic Processes', 259), ('Digestive System Diseases', 230), ('Skin and Connective Tissue Diseases', 229), ('Neoplasms by Histologic Type', 165), ('Gastrointestinal Diseases', 153), ('Nervous System Diseases', 151), ('Skin Diseases', 147), ('Disease Attributes', 145), ('Nutritional and Metabolic Diseases', 143), ('Autoimmune Diseases', 138), ('Respiratory Tract Neoplasms', 135), ('Hemic and Lymphatic Diseases', 133), ('Infections', 133), ('Thoracic Neoplasms', 133), ('Cardiovascular Diseases', 131), ('Carcinoma, Bronchogenic', 129), ('Bronchial Neoplasms', 129), ('Urogenital Diseases', 129), ('Intestinal Diseases', 124), ('Chronic Disease', 123), ('Signs and Symptoms', 122), ('Endocrine System Diseases', 122), ('Metabolic Diseases', 116), ('Lung Neoplasms', 114)]
[('Carcinoma, Non-

In [6]:
# studies with no disease category = 194. typos or too general or even terms that are not diseases

no_mesh = []
for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    cbm = s.get("derivedSection", {}).get("conditionBrowseModule", {})
    if not cbm.get("meshes"):
        conds = s["protocolSection"].get("conditionsModule", {}).get("conditions")
        no_mesh.append((nct, conds))

print(len(no_mesh))
for item in no_mesh[:10]:
    print(item)

194
('NCT04090411', ['Moderate to Severe Ulcerative Colitis'])
('NCT07533955', ['Rhematoid Arthritis'])
('NCT02413580', ['Myasthenia Gravis Exacerbations'])
('NCT04452318', ['Healthy Participants'])
('NCT06978010', ['Cesarean Section Wound', 'Platelet-rich Plasma (PRP)', 'Regenerative Medicine'])
('NCT06452277', ['Advanced Non-small Cell Lung Cancer, HER2 Mutation'])
('NCT03143153', ['Various Advanced Cancer'])
('NCT06834360', ['Chronic Rhinosinusitis With Nasal Polyps'])
('NCT03110107', ['Advanced Cancer'])
('NCT05962554', ['Unemployment'])


In [7]:
#test for choosing area list. how many studies are within chosen therap areas = 1449. terms for those 74 uncovered areas.
candidate_terms = {
    "Oncology":                   ["Neoplasms"],
    "Respiratory":                ["Respiratory Tract Diseases"],
    "Immune system":              ["Immune System Diseases"],
    "Skin and connective tissue": ["Skin and Connective Tissue Diseases"],
    "Digestive":                  ["Digestive System Diseases"],
    "Nervous system":             ["Nervous System Diseases"],
    "Metabolic and nutritional":  ["Nutritional and Metabolic Diseases"],
    "Cardiovascular":             ["Cardiovascular Diseases"],
    "Blood and lymphatic":        ["Hemic and Lymphatic Diseases"],
    "Infections":                 ["Infections"],
    "Urogenital":                 ["Urogenital Diseases"],
    "Endocrine":                  ["Endocrine System Diseases"],
    "Musculoskeletal":            ["Musculoskeletal Diseases"],
    "Mental health":              ["Mental Disorders"],
}

area_counts = Counter()
uncovered_terms = Counter()
n_with_terms = 0
n_covered = 0

for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    cbm = s.get("derivedSection", {}).get("conditionBrowseModule", {})
    terms = set(m["term"] for m in cbm.get("meshes", [])) | set(a["term"] for a in cbm.get("ancestors", []))
    if not terms:
        continue
    n_with_terms += 1
    found = [area for area, names in candidate_terms.items() if terms & set(names)]
    area_counts.update(found)
    if found:
        n_covered += 1
    else:
        uncovered_terms.update(terms)

print(n_with_terms, n_covered)
print(area_counts.most_common())
print(uncovered_terms.most_common(25))

1523 1449
[('Oncology', 433), ('Respiratory', 299), ('Immune system', 288), ('Digestive', 234), ('Skin and connective tissue', 230), ('Nervous system', 153), ('Cardiovascular', 147), ('Metabolic and nutritional', 144), ('Infections', 137), ('Blood and lymphatic', 133), ('Endocrine', 129), ('Urogenital', 129), ('Musculoskeletal', 112), ('Mental health', 93)]
[('Behavior', 27), ('Pathological Conditions, Signs and Symptoms', 22), ('Pathologic Processes', 13), ('Signs and Symptoms', 12), ('Eye Diseases', 12), ('Social Behavior', 11), ('Emotional Regulation', 9), ('Self-Control', 9), ('Behavioral Symptoms', 9), ('Neurologic Manifestations', 8), ('Pain', 8), ('Retinal Diseases', 7), ('Stomatognathic Diseases', 7), ('Personal Satisfaction', 6), ('Psychological Well-Being', 6), ('Disease Attributes', 5), ('Tooth Diseases', 5), ('Congenital, Hereditary, and Neonatal Diseases and Abnormalities', 5), ('Infant, Newborn, Diseases', 5), ('Retinopathy of Prematurity', 5), ('Infant, Premature, Diseas

In [8]:
#study_area final version. dict with 13 therap areas

area_terms = {
    "Oncology":                   ["Neoplasms"],
    "Respiratory":                ["Respiratory Tract Diseases"],
    "Immune system":              ["Immune System Diseases"],
    "Digestive":                  ["Digestive System Diseases"],
    "Skin and connective tissue": ["Skin and Connective Tissue Diseases"],
    "Nervous system":             ["Nervous System Diseases"],
    "Cardiovascular":             ["Cardiovascular Diseases"],
    "Metabolic and endocrine":    ["Nutritional and Metabolic Diseases", "Endocrine System Diseases"],
    "Infections":                 ["Infections"],
    "Blood and lymphatic":        ["Hemic and Lymphatic Diseases"],
    "Urogenital":                 ["Urogenital Diseases"],
    "Musculoskeletal":            ["Musculoskeletal Diseases"],
    "Mental health":              ["Mental Disorders"],
}

area_rows = []
for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    cbm = s.get("derivedSection", {}).get("conditionBrowseModule", {})
    terms = set(m["term"] for m in cbm.get("meshes", [])) | set(a["term"] for a in cbm.get("ancestors", []))
    found = [area for area, names in area_terms.items() if terms & set(names)]
    if not found:
        found = ["Not classified"]
    for area in found:
        area_rows.append({"nctId": nct, "area": area})

df_area = pd.DataFrame(area_rows)

print(df_area.shape)
print(df_area["nctId"].nunique())
print(df_area.duplicated().sum())
print(df_area["area"].value_counts())

(2849, 2)
1717
0
area
Oncology                      433
Respiratory                   299
Immune system                 288
Not classified                268
Digestive                     234
Skin and connective tissue    230
Metabolic and endocrine       193
Nervous system                153
Cardiovascular                147
Infections                    137
Blood and lymphatic           133
Urogenital                    129
Musculoskeletal               112
Mental health                  93
Name: count, dtype: int64


In [9]:
df_area.to_csv("../data/clean/study_area.csv", index=False)

check = pd.read_csv("../data/clean/study_area.csv")
print(check.shape == df_area.shape)     

True


In [10]:
# no phase intervention
s = all_studies[0]
print(s["protocolSection"]["armsInterventionsModule"].keys())
print(s["protocolSection"]["armsInterventionsModule"]["interventions"])

dict_keys(['armGroups', 'interventions'])
[{'type': 'DRUG', 'name': 'PF-04965842 100 mg', 'description': 'PF-04965842 100 mg, administered as two tablets to be taken orally once daily for 40 weeks', 'armGroupLabels': ['PF-04965842 100 mg QD'], 'otherNames': ['Abrocitinib']}, {'type': 'DRUG', 'name': 'PF-04965842 200 mg', 'description': 'PF-04965842 200 mg, administered as two tablets to be taken orally once daily for 40 weeks', 'armGroupLabels': ['PF-04965842 200 mg QD'], 'otherNames': ['Abrocitinib']}, {'type': 'DRUG', 'name': 'Placebo', 'description': 'Placebo, administered as two tablets to be taken orally once daily for 40 weeks', 'armGroupLabels': ['Placebo QD']}]


In [11]:
# "phase of ID" and no phase intervention

phase_of = dict(zip(df_int["nctId"], df_int["phaseLabel"]))

types_nophase = Counter()
types_phase = Counter()
n_nophase = 0
n_nophase_empty = 0

for s in all_studies:
    nct = s["protocolSection"]["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    interventions = s["protocolSection"].get("armsInterventionsModule", {}).get("interventions", [])
    types = set(i.get("type") for i in interventions)
    if phase_of[nct] == "No phase":
        n_nophase += 1
        if not types:
            n_nophase_empty += 1
        types_nophase.update(types)
    else:
        types_phase.update(types)

print(n_nophase, n_nophase_empty)
print(types_nophase.most_common())
print(types_phase.most_common())

287 0
[('DEVICE', 88), ('BEHAVIORAL', 76), ('OTHER', 71), ('PROCEDURE', 35), ('DIETARY_SUPPLEMENT', 25), ('DRUG', 16), ('DIAGNOSTIC_TEST', 12), ('COMBINATION_PRODUCT', 4), ('BIOLOGICAL', 2), ('RADIATION', 1)]
[('DRUG', 1320), ('BIOLOGICAL', 182), ('OTHER', 156), ('PROCEDURE', 15), ('COMBINATION_PRODUCT', 14), ('DEVICE', 11), ('RADIATION', 10), ('DIAGNOSTIC_TEST', 3), ('DIETARY_SUPPLEMENT', 2), ('GENETIC', 2), ('BEHAVIORAL', 1)]
